# Voice notice translator

Record a school or office announcement once, in any Indian language, and get it back as text **and** audio in every language your audience speaks.

Pipeline: `indic-transcribe` → `indic-translate` → `indic-speak`.

**Requires** `BODHAN_API_KEY` in `.env` (copy `.env.example`).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, json, requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
HEADERS = {"Authorization": f"Bearer {os.environ['BODHAN_API_KEY']}"}
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)


def bodhan(method: str, path: str, **kw):
    resp = requests.request(method, f"{BASE_URL}{path}", headers={**HEADERS, **kw.pop("headers", {})}, timeout=120, **kw)
    if not resp.ok:
        try:
            err = resp.json()["error"]
            raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
        except (ValueError, KeyError):
            resp.raise_for_status()
    return resp

## 1. Transcribe the source recording

The repo's sample clip is Hindi. Drop your own ~30 s WAV/MP3 into `sample_data/` and change the path.

In [ ]:
SOURCE = "../../sample_data/sample_hindi.wav"
SOURCE_LANG = "hi"

with open(SOURCE, "rb") as fh:
    transcript = bodhan("POST", "/v1/audio/transcriptions", files={"file": fh},
                        data={"model": "indic-transcribe", "language": SOURCE_LANG}).json()["text"]
transcript

## 2. Translate into every target language

In [ ]:
# language -> (voice, display name)
TARGETS = {
    "en": ("Amit", "English"),
    "ta": ("Anitha", "Tamil"),
    "bn": ("Ishita", "Bengali"),
    "te": ("Sravani", "Telugu"),
    "mr": ("Anagha", "Marathi"),
}

translations = {}
for lang in TARGETS:
    translations[lang] = bodhan("POST", "/translate", json={"text": transcript, "target_language": lang}).json()["translation"]
    print(f"{TARGETS[lang][1]:8} {translations[lang]}")

## 3. Speak each translation

In [ ]:
JSON = {"Content-Type": "application/json"}
for lang, text in translations.items():
    voice, name = TARGETS[lang]
    wav = bodhan("POST", "/v1/audio/speech", headers=JSON, json={
        "model": "indic-speak", "input": text, "voice": voice,
        "instructions": json.dumps({"lang": lang, "style": "AIR style news"}),
    }).content
    path = OUT / f"notice_{lang}.wav"
    path.write_bytes(wav)
    print(f"{name:8} -> {path} ({len(wav) // 1024} KB)")

In [ ]:
from IPython.display import Audio, display
for lang in TARGETS:
    print(TARGETS[lang][1])
    display(Audio(str(OUT / f"notice_{lang}.wav")))

## 4. Ship it

`outputs/` now holds one WAV per language plus you have the text in `translations`. Ideas:

- Write a `notice.json` with `{lang: {text, audio_path}}` for a web page or WhatsApp bot to consume.
- Long announcements: chunk the source audio (see `getting-started/speech-to-text`) and split translations by sentence before `indic-speak`.
- Swap `"AIR style news"` for `"Customer Care"` when the announcement is a helpdesk message.